# Sprint 7F Family-Aware Target-Observation Context Encoder Runner

Colab is a runner only. Model, encoder logic, metrics, diagnostics, figures, and reports are implemented in repository code.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail
REPO_URL="${REPO_URL:-https://github.com/YasinEkici/crispr-gnn-offtarget.git}"
BRANCH="${BRANCH:-sprint7/gat-gatv2}"
cd /content
if [ ! -d crispr-gnn-offtarget/.git ]; then
  git clone --branch "$BRANCH" "$REPO_URL" crispr-gnn-offtarget
else
  cd crispr-gnn-offtarget
  git fetch origin "$BRANCH"
  git checkout "$BRANCH"
  git pull --ff-only origin "$BRANCH"
fi
cd /content/crispr-gnn-offtarget
git status --short --branch

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
python -m pip install -q uv
uv sync

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT="${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
ALT_DRIVE_ROOT="/content/drive/MyDrive/crispr-gnn-offtarget"
if [ ! -d "$DRIVE_ROOT" ] && [ -d "$ALT_DRIVE_ROOT" ]; then
  DRIVE_ROOT="$ALT_DRIVE_ROOT"
fi
echo "Using DRIVE_ROOT=$DRIVE_ROOT"
test -d "$DRIVE_ROOT"
mkdir -p data/raw data/processed
if [ -d "$DRIVE_ROOT/data/raw" ]; then
  rsync -a "$DRIVE_ROOT/data/raw/" data/raw/
fi
if [ -d "$DRIVE_ROOT/data/processed" ]; then
  rsync -a "$DRIVE_ROOT/data/processed/" data/processed/
fi
if [ ! -d data/processed/graphs/sprint5b/graph_c_context_observation ]; then
  echo "Building Sprint 5B Graph C S5F2 artifact required by Sprint 7F..."
  uv run python scripts/build_sprint5b_graph_c_energy_features.py
fi
test -d data/processed/graphs/sprint5b/graph_c_context_observation
find data/processed/graphs -maxdepth 3 -type f -name 'manifest.json' | sort

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
RUN_ID="sprint7f_target_context_encoder_seed42_$(date -u +%Y%m%d_%H%M%S)"
uv run python scripts/run_sprint7f_target_context_encoder.py \
  --config configs/sweeps/sprint7f_target_context_encoder.yaml \
  --run-id "$RUN_ID"
echo "$RUN_ID" > outputs/sprint7f/latest_run_id.txt

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
test -f outputs/sprint7f/target_context_encoder_comparison.csv
test -f outputs/sprint7f/target_context_encoder_report.md
test -f outputs/sprint7f/target_context_encoder_run_manifest.json
test -f outputs/sprint7f/diagnostics/target_context_encoder_audit.csv
test -f outputs/sprint7f/figures/target_context_encoder_auprc_comparison.png
test -f outputs/sprint7f/figures/target_context_encoder_attention_by_edge_kind.png
uv run python - <<'PY'
import pandas as pd
results = pd.read_csv('outputs/sprint7f/target_context_encoder_comparison.csv')
expected = {
    'S7F_R1_unified_deep_context_encoder',
    'S7F_R2_family_aware_context_encoder',
    'S7F_R3_family_aware_experimental_emphasis',
}
missing = expected - set(results['predeclared_run_id'].astype(str))
if missing:
    raise SystemExit(f'missing Sprint 7F run ids: {sorted(missing)}')
audit = pd.read_csv('outputs/sprint7f/diagnostics/target_context_encoder_audit.csv')
if set(audit.loc[audit['split'].isin(['train','val','test']), 'context_edges_used']) != {0}:
    raise SystemExit('Sprint 7F canonical runs must drop context edges')
audit_contract = audit.loc[audit['split'].isin(['train','val','test'])]
if audit_contract['candidate_attention_attr_abs_sum'].min() <= 0:
    raise SystemExit('candidate S5F2 attention attrs must remain active')
print(results[['predeclared_run_id','target_context_encoder_type','test_auprc','test_mcc','test_specificity','test_tn','test_fp']].to_string(index=False))
PY

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT="${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
ALT_DRIVE_ROOT="/content/drive/MyDrive/crispr-gnn-offtarget"
if [ ! -d "$DRIVE_ROOT" ] && [ -d "$ALT_DRIVE_ROOT" ]; then
  DRIVE_ROOT="$ALT_DRIVE_ROOT"
fi
RUN_ID="$(cat outputs/sprint7f/latest_run_id.txt)"
DEST="$DRIVE_ROOT/returned_outputs/$RUN_ID"
mkdir -p "$DEST"
rsync -a --exclude='model.pt' --exclude='.DS_Store' outputs/sprint7f/ "$DEST/"
echo "Copied Sprint 7F outputs to $DEST"